# MNIST Classification With Learned Superposition Activations

This notebook runs on the **full MNIST** split from sklearn/OpenML (~70k samples). The first run downloads MNIST and caches it locally.

Training uses **JAX + Optax** when available (`pip install "qfun[gpu]"`, then install a [CUDA-enabled jaxlib](https://jax.readthedocs.io/en/latest/installation.html) on Linux or Windows for GPU). The same code path runs on CPU if only the CPU jaxlib is installed. Without JAX, set `use_jax=False` in the config (PennyLane autograd over the full set can be very slow).

Features are **standardized** and compressed with **PCA** before baselines and quantum models.

All figures, console output, and numeric summaries are written under ``notebooks/note10_outputs/<run_id>/``.



In [ ]:
import atexit
import csv
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

for _p in (Path.cwd(), Path.cwd().parent):
    if (_p / "qfun").is_dir():
        _root = str(_p.resolve())
        if _root not in sys.path:
            sys.path.insert(0, _root)
        break

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
_THIS = Path.cwd().resolve()
if not (_THIS / "qfun").is_dir():
    _THIS = _THIS.parent
OUTPUT_ROOT = _THIS / "notebooks" / "note10_outputs" / RUN_ID
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
BASELINES_DIR = OUTPUT_ROOT / "baselines"
QUANTUM_DIR = OUTPUT_ROOT / "quantum"


class _Tee:
    """Mirror writes to multiple text streams (e.g. terminal + run log)."""

    def __init__(self, *streams):
        self.streams = streams

    def write(self, data: str) -> None:
        for s in self.streams:
            s.write(data)
            s.flush()

    def flush(self) -> None:
        for s in self.streams:
            s.flush()


_console_log = open(OUTPUT_ROOT / "console.log", "w", encoding="utf-8")
sys.stdout = _Tee(sys.__stdout__, _console_log)
sys.stderr = _Tee(sys.__stderr__, _console_log)


def _note10_restore_stdio() -> None:
    """Undo Tee and close log so interpreter shutdown does not write to a closed file."""
    sys.stdout = sys.__stdout__
    sys.stderr = sys.__stderr__
    if not _console_log.closed:
        _console_log.close()


atexit.register(_note10_restore_stdio)

import matplotlib.pyplot as plt
import numpy as np

_plt_show = plt.show
_figure_counter = 0


def _show_and_savefig(*args, **kwargs):
    """Keep default display behavior and also save the current figure to OUTPUT_ROOT."""
    global _figure_counter
    fig = plt.gcf()
    if fig.get_axes():
        _figure_counter += 1
        out = OUTPUT_ROOT / f"fig_{_figure_counter:03d}.png"
        fig.savefig(out, dpi=200, bbox_inches="tight")
    return _plt_show(*args, **kwargs)


plt.show = _show_and_savefig

from qfun.datasets import load_classification_dataset, prepare_classification_split
from qfun.qfan._classification_benchmarks import (
    BaselineResult,
    QuantumExperimentResult,
    build_comparison_rows,
    display_baseline_suite,
    display_quantum_result,
    print_comparison_table,
    print_split_summary,
    plot_training_diagnostics,
    run_default_baseline_suite,
    run_quantum_experiment,
)

print(f"Run artifacts directory: {OUTPUT_ROOT}", flush=True)


def _save_json(path: Path, obj: object) -> None:
    path.write_text(json.dumps(obj, indent=2), encoding="utf-8")


def _save_baseline_artifacts(results: dict[str, BaselineResult]) -> None:
    BASELINES_DIR.mkdir(parents=True, exist_ok=True)
    rows = []
    for key, res in results.items():
        sub = BASELINES_DIR / key.replace(" ", "_")
        sub.mkdir(exist_ok=True)
        np.save(sub / "confusion_matrix.npy", res.confusion_matrix)
        (sub / "classification_report.txt").write_text(res.classification_report, encoding="utf-8")
        rows.append(
            {
                "key": key,
                "name": res.name,
                "accuracy": res.accuracy,
                "macro_f1": res.macro_f1,
            }
        )
    _save_json(BASELINES_DIR / "summary.json", rows)


def _save_quantum_artifacts(tag: str, result: QuantumExperimentResult) -> None:
    d = QUANTUM_DIR / tag
    d.mkdir(parents=True, exist_ok=True)
    np.save(d / "losses.npy", result.losses)
    np.save(d / "confusion_matrix.npy", result.confusion_matrix)
    (d / "classification_report.txt").write_text(result.classification_report, encoding="utf-8")
    train_steps = [s.step for s in result.training_snapshots]
    train_losses = [s.loss for s in result.training_snapshots]
    np.savez(
        d / "training_snapshots.npz",
        steps=np.asarray(train_steps),
        losses=np.asarray(train_losses),
    )
    if result.accuracy_history:
        np.savez(
            d / "accuracy_history.npz",
            steps=np.asarray([s.step for s in result.accuracy_history], dtype=np.int32),
            train_accuracy=np.asarray([s.train_accuracy for s in result.accuracy_history]),
            test_accuracy=np.asarray([s.test_accuracy for s in result.accuracy_history]),
        )
    _save_json(
        d / "summary.json",
        {
            "label": result.label,
            "mode": result.mode,
            "train_accuracy": result.train_accuracy,
            "test_accuracy": result.test_accuracy,
            "macro_f1": result.macro_f1,
            "representative_units": list(result.representative_units),
            "eval_shots": result.eval_shots,
            "snapshot_interval": result.snapshot_interval,
        },
    )




## Config

Defaults mirror notebook 08. `use_jax=True` enables minibatch Adam on JAX (CPU or GPU). Without JAX installed, set `use_jax=False` (expect long runtimes on ~52k training points). Use `log_every=1` for a loss line every epoch and `show_training_progress=True` with `tqdm` installed for a live minibatch bar (JAX) or epoch bar (PennyLane).




In [ ]:
data_seed = 7
test_size = 0.2
pca_components = 32

hidden_units = 6
n_qubits = 5
steps = 100
learning_rate = 0.01
log_every = 1  # print train loss every epoch; set higher to reduce console noise
show_training_progress = True  # tqdm bar if installed: pip install tqdm
snapshot_interval = 10
eval_shots = 8_000

try:
    import jax  # noqa: F401

    use_jax = True
except ImportError:
    use_jax = False
    print('Tip: pip install "qfun[gpu]" for JAX training on full MNIST (much faster).')

batch_size = 1024

_save_json(
    OUTPUT_ROOT / "config.json",
    {
        "data_seed": data_seed,
        "test_size": test_size,
        "pca_components": pca_components,
        "hidden_units": hidden_units,
        "n_qubits": n_qubits,
        "steps": steps,
        "learning_rate": learning_rate,
        "log_every": log_every,
        "show_training_progress": show_training_progress,
        "snapshot_interval": snapshot_interval,
        "eval_shots": eval_shots,
        "use_jax": use_jax,
        "batch_size": batch_size,
        "run_id": RUN_ID,
    },
)



## 1. Load, standardize, and compress MNIST

`load_classification_dataset("mnist")` may download once via sklearn/OpenML. We stratify the train/test split on labels, **standardize** 784-dimensional pixels, and apply **PCA** to `pca_components` dimensions before any model training.




In [ ]:
mnist_dataset = load_classification_dataset("mnist")
mnist_split = prepare_classification_split(
    mnist_dataset,
    test_size=test_size,
    seed=data_seed,
    standardize=True,
    pca_components=pca_components,
)
class_names = mnist_split.target_names
print(f"Dataset size: {mnist_dataset.X.shape[0]} samples (full MNIST)")
print_split_summary(mnist_dataset.name, mnist_split)

_save_json(
    OUTPUT_ROOT / "dataset_split.json",
    {
        "name": mnist_dataset.name,
        "n_total": int(mnist_dataset.X.shape[0]),
        "n_train": int(mnist_split.x_train.shape[0]),
        "n_test": int(mnist_split.x_test.shape[0]),
        "n_features": int(mnist_split.x_train.shape[1]),
        "class_names": list(class_names),
        "train_class_counts": np.bincount(mnist_split.y_train).tolist(),
        "test_class_counts": np.bincount(mnist_split.y_test).tolist(),
    },
)
np.savez_compressed(
    OUTPUT_ROOT / "data_split_arrays.npz",
    x_train=mnist_split.x_train.astype(np.float64),
    y_train=mnist_split.y_train.astype(np.int32),
    x_test=mnist_split.x_test.astype(np.float64),
    y_test=mnist_split.y_test.astype(np.int32),
)



## 2. Baselines




In [ ]:
baseline_results = run_default_baseline_suite(
    mnist_split,
    seed=data_seed,
    mlp_hidden_layer_sizes=(64,),
)
display_baseline_suite(baseline_results, class_names)
_save_baseline_artifacts(baseline_results)



## 3. Standard Superposition Activations




In [ ]:
standard_result = run_quantum_experiment(
    "standard",
    label="MNIST standard superposition activations",
    split=mnist_split,
    hidden_units=hidden_units,
    n_qubits=n_qubits,
    steps=steps,
    learning_rate=learning_rate,
    seed=data_seed,
    log_every=log_every,
    snapshot_interval=snapshot_interval,
    eval_shots=eval_shots,
    use_jax=use_jax,
    batch_size=batch_size,
    show_training_progress=show_training_progress,
)
display_quantum_result(standard_result, class_names)
_save_quantum_artifacts("standard", standard_result)



### Standard Training Process (Snapshots)




In [ ]:
plot_training_diagnostics(standard_result)



## 4. Mode A Signed Superposition Activations




In [ ]:
mode_a_result = run_quantum_experiment(
    "mode_a",
    label="MNIST Mode A superposition activations",
    split=mnist_split,
    hidden_units=hidden_units,
    n_qubits=n_qubits,
    steps=steps,
    learning_rate=learning_rate,
    seed=data_seed,
    log_every=log_every,
    snapshot_interval=snapshot_interval,
    eval_shots=eval_shots,
    use_jax=use_jax,
    batch_size=batch_size,
    show_training_progress=show_training_progress,
)
display_quantum_result(mode_a_result, class_names)
_save_quantum_artifacts("mode_a", mode_a_result)



### Mode A Training Process (Snapshots)




In [ ]:
plot_training_diagnostics(mode_a_result)



## 5. Mode B Signed Superposition Activations




In [ ]:
mode_b_result = run_quantum_experiment(
    "mode_b",
    label="MNIST Mode B superposition activations",
    split=mnist_split,
    hidden_units=hidden_units,
    n_qubits=n_qubits,
    steps=steps,
    learning_rate=learning_rate,
    seed=data_seed,
    log_every=log_every,
    snapshot_interval=snapshot_interval,
    eval_shots=eval_shots,
    use_jax=use_jax,
    batch_size=batch_size,
    show_training_progress=show_training_progress,
)
display_quantum_result(mode_b_result, class_names)
_save_quantum_artifacts("mode_b", mode_b_result)



### Mode B Training Process (Snapshots)




In [ ]:
plot_training_diagnostics(mode_b_result)



## 6. Final Comparison




In [ ]:
comparison_rows = build_comparison_rows(
    baseline_results,
    [standard_result, mode_a_result, mode_b_result],
)
print_comparison_table(comparison_rows)

with open(OUTPUT_ROOT / "comparison.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["Model", "Test accuracy", "Macro-F1"])
    w.writerows(comparison_rows)

_save_json(
    OUTPUT_ROOT / "comparison.json",
    [{"model": name, "test_accuracy": acc, "macro_f1": f1} for name, acc, f1 in comparison_rows],
)

print(f"Finished. Figures: fig_*.png, log: console.log, metrics under subfolders of {OUTPUT_ROOT}", flush=True)
